# Coding RAG

Retrieval-Augmented Generation for codebases differs from document RAG: structure (AST), build graphs, and symbol identity matter as much as embedding similarity. This notebook covers indexing, hybrid retrieval, query strategies, and prompt packing.


## Learning Objectives

- Explain why coding RAG differs from doc RAG
- Build AST-aware chunking for Python
- Implement a tiny hybrid retriever
- Pack context under a token budget with citations


## 1. Why Coding RAG Is Different

| Doc RAG | Coding RAG |
|---------|------------|
| Paragraph chunks | Functions/classes/blocks |
| Semantic similarity | + exact symbol match |
| Citations optional | Paths/lines critical |
| Static corpus | Rapidly changing git tree |
| One language prose | Polyglot + configs |

### Pitfalls
Embedding-only search misses exact identifiers (`UserServiceImpl`).


## 2. Indexing Pipeline

```mermaid
flowchart LR
  Repo --> Parse[Parse / Tree-sitter / AST]
  Parse --> Chunk[Symbol chunks]
  Chunk --> Meta[Path / symbol / imports meta]
  Meta --> Dense[Embeddings]
  Meta --> Sparse[BM25 / keywords]
  Dense --> Index[(Hybrid index)]
  Sparse --> Index
```


In [ ]:
# Demo 1 — AST-aware Python chunking
import ast
from dataclasses import dataclass

@dataclass
class Chunk:
    path: str
    name: str
    kind: str
    start: int
    end: int
    text: str

def chunk_python(path: str, source: str) -> list[Chunk]:
    tree = ast.parse(source)
    lines = source.splitlines()
    out: list[Chunk] = []
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            start, end = node.lineno, node.end_lineno or node.lineno
            text = "\n".join(lines[start-1:end])
            kind = "class" if isinstance(node, ast.ClassDef) else "function"
            out.append(Chunk(path, node.name, kind, start, end, text))
    return out

src = "x=1\n\nclass A:\n    def m(self):\n        return 1\n\ndef f():\n    return 2\n"
for c in chunk_python("mod.py", src):
    print(c.kind, c.name, f"L{c.start}-{c.end}")


In [ ]:
# Demo 2 — Tiny hybrid retrieval: keyword overlap + fake dense score
from collections import Counter
import math, re

def tokenize(s: str) -> list[str]:
    return re.findall(r"[A-Za-z_][A-Za-z0-9_]*", s.lower())

def hybrid_score(query: str, doc: str, dense: float, alpha: float = 0.6) -> float:
    q, d = tokenize(query), tokenize(doc)
    if not q or not d:
        return dense
    qc, dc = Counter(q), Counter(d)
    overlap = sum((qc & dc).values())
    sparse = overlap / math.sqrt(len(q) * len(set(d)))
    return alpha * dense + (1 - alpha) * sparse

docs = {
    "auth.py": "class AuthService: def login(self, user): ...",
    "billing.py": "class BillingService: def charge(self): ...",
}
query = "AuthService login"
for p, t in docs.items():
    print(p, round(hybrid_score(query, t, dense=0.1 if "Bill" in p else 0.5), 3))


## 3. Query Strategies

1. **Symbol-first** — exact/fuzzy definition lookup
2. **Semantic** — “where do we validate JWTs?”
3. **Error-driven** — stack frames as queries
4. **Multi-hop** — definition → callers → tests
5. **Config-aware** — include `pyproject`, OpenAPI, proto


## 4. Prompt Packing

Pack order heuristic: tests + definitions + callers + style examples, with path headers and budgets.


In [ ]:
# Demo 3 — Context packer with token budget and citations
from dataclasses import dataclass

@dataclass
class Hit:
    path: str
    start: int
    end: int
    text: str
    score: float

def est_tokens(s: str) -> int:
    return max(1, len(s) // 4)

def pack_hits(hits: list[Hit], budget: int) -> str:
    parts = []
    used = 0
    for h in sorted(hits, key=lambda x: -x.score):
        header = f"// {h.path}:{h.start}-{h.end}\n"
        block = header + h.text + "\n"
        cost = est_tokens(block)
        if used + cost > budget:
            continue
        parts.append(block)
        used += cost
    return "\n".join(parts)

hits = [
    Hit("a.py", 1, 5, "def f():\n    return 1\n", 0.9),
    Hit("b.py", 10, 20, "def g():\n    return f()\n" * 5, 0.7),
]
print(pack_hits(hits, 80))


### Try it yourself — Coding RAG

- Chunk a real package by AST and measure average chunk tokens
- Compare BM25-only vs hybrid on 20 symbol queries
- Add import-graph expansion: retrieve definition + direct callers


## Glossary / Key Terms

| Term | Meaning |
|------|--------|
| `AST chunking` | Split code by syntax tree nodes |
| `Hybrid retrieval` | Combine sparse + dense scores |
| `Multi-hop` | Follow references across multiple retrieves |


## Deep Dive Workshop — 07 Coding Rag

This section expands the notebook into instructor/textbook depth. Work through each subsection: **definition → why it matters → how it works → intuition → pitfalls → when to use**.

```mermaid
flowchart TB
  D[Definition] --> W[Why it matters]
  W --> H[How it works]
  H --> I[Intuition]
  I --> P[Pitfalls]
  P --> U[When to use]
```


### Concept card pack for `07-coding-rag`

| Concept | Definition | Why it matters | Common pitfall |
|---------|------------|----------------|----------------|
| Primary abstraction | Core object this lesson centers on | Anchors design conversations | Vague naming |
| Quality oracle | How you know the system is right | Prevents demo-driven development | Using vibes only |
| Latency budget | Max user-visible wait | Drives architecture | Ignoring TTFT vs e2e |
| Cost unit | $ per successful task | Makes tradeoffs real | Optimizing tokens not outcomes |
| Trust boundary | Where data/control changes hands | Security design | Treating vendors as internal |
| Feedback loop | How production improves the system | Sustainable quality | No path from thumbs-down to evals |

**Intuition:** If you cannot fill this table for your system, you are not ready to choose models or frameworks.


### Pipeline walkthrough (apply to 07-coding-rag)

```
1. Input arrives (user / job / webhook)
2. Normalize + authorize + budget check
3. Gather context (files, RAG, tools, memory)
4. Model / deterministic compute
5. Validate output (schema, policy, tests)
6. Side effects (write, ticket, PR) with authz
7. Observe (metrics, traces, feedback)
8. Learn (eval suite growth, prompt/model revision)
```

**When to compress steps:** tiny internal tools. **When to keep all steps:** multi-tenant or regulated production.


### Coding-models advanced notes

**Fill-in-the-middle formats** differ by vendor; always keep an adapter layer.  
**Repo agents** should treat tests as the north star and protect test files by default.  
**Coding RAG** should combine symbol lookup + BM25 + embeddings; embeddings alone miss identifiers.

| Task | Prefer | Avoid |
|------|--------|-------|
| Ghost text | FIM-capable small/fast model | Giant chat model sync |
| API migration | Agent + tests | Single-shot whole-repo rewrite |
| Explain legacy | Chat + citations | Uncited summaries |


In [ ]:
# Extra demo — diff extraction toy for coding assistants
import re

def extract_fenced_blocks(text: str) -> list[tuple[str, str]]:
    pat = re.compile(r"```(\w+)?\n(.*?)```", re.S)
    return [(m.group(1) or 'txt', m.group(2)) for m in pat.finditer(text)]

sample = '''Here is a fix:\n```python\ndef add(a,b):\n    return a+b\n```\n'''
print(extract_fenced_blocks(sample))


In [ ]:
# Extra demo — simple symbol index for coding RAG
import ast
from collections import defaultdict

def index_symbols(source: str, path: str) -> dict[str, list[str]]:
    tree = ast.parse(source)
    idx = defaultdict(list)
    for n in ast.walk(tree):
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            idx[n.name].append(f"{path}:{n.lineno}")
    return dict(idx)

print(index_symbols('class Foo:\n  def bar(self):\n    pass\n', 'a.py'))


### Sample interview Q&A — coding models

**Q:** Copilot-quality inline completion is slow. What do you do?  
**A:** Separate completion model from chat model; shrink context to locals + imports; consider speculative decoding / smaller quantized model; measure acceptance rate not just tok/s.

**Q:** How do you evaluate a coding assistant for a monorepo?  
**A:** Private suite: completion acceptance, unit-test pass on generated patches, security scanner findings, and human review on a stratified sample of PR diffs.


### Comparison matrix exercise

Fill this for two competing designs in this topic:

| Dimension | Option A | Option B | Winner / why |
|-----------|----------|----------|--------------|
| Latency | | | |
| Cost at 10× scale | | | |
| Quality risk | | | |
| Ops burden | | | |
| Security / privacy | | | |
| Time to MVP | | | |


In [ ]:
# Workshop demo — decision scorecard
from dataclasses import dataclass

@dataclass
class Option:
    name: str
    latency: int  # 1=best .. 5=worst
    cost: int
    quality_risk: int
    ops: int
    security: int

def score(o: Option, weights=None) -> float:
    weights = weights or dict(latency=1, cost=1, quality_risk=2, ops=1, security=2)
    return (
        o.latency*weights['latency'] + o.cost*weights['cost'] +
        o.quality_risk*weights['quality_risk'] + o.ops*weights['ops'] +
        o.security*weights['security']
    )

a = Option('A', 2, 3, 2, 2, 2)
b = Option('B', 3, 1, 3, 4, 2)
print(a.name, score(a), b.name, score(b), '-> prefer', a.name if score(a)<score(b) else b.name)


In [ ]:
# Workshop demo — experiment log (use while studying this notebook)
from dataclasses import dataclass, asdict
import json, time

@dataclass
class Experiment:
    hypothesis: str
    setup: str
    metric: str
    baseline: float | None = None
    treatment: float | None = None
    notes: str = ''
    ts: float = 0.0

    def __post_init__(self):
        if not self.ts:
            self.ts = time.time()

exp = Experiment(
    hypothesis='Technique from this lesson improves the primary metric',
    setup='Describe fixtures / model / dataset version',
    metric='name of metric',
    baseline=0.0,
    treatment=0.0,
)
print(json.dumps(asdict(exp), indent=2))


### ASCII architecture sketch template

```
[ Clients ]
     |
[ Edge / API Gateway ] -- authn/z, rate limit
     |
[ Orchestration ] ------+-- prompts / policies
     |                  +-- eval hooks
     +-- context layer (RAG / tools / memory)
     |
[ Model interface ] ---- local and/or cloud
     |
[ Data plane ] --------- indexes, OLTP, object store
     |
[ Observability ] ------ logs, metrics, traces, feedback
```

Copy into your notes and annotate trust boundaries with `***`.


### Pitfalls clinic (read aloud)

1. **Metric theater** — optimizing a proxy that users don't feel  
2. **Context stuffing** — more tokens ≠ more truth  
3. **Prompt as security** — never the only control  
4. **Hidden coupling** — tools/models/indexes version-drift  
5. **No rollback** — can't revert prompt/model quickly  
6. **Eval contamination** — testing on training-like snippets  
7. **Happy-path demos** — skipping adversarial & empty-retrieve cases  


### Try it yourself — extended set

1. Teach the top 3 ideas from this notebook to a rubber duck in 5 minutes  
2. Write 5 quiz questions (with answers) for a junior engineer  
3. Implement one code demo with a real dependency (API or local model) using env placeholders  
4. Break a naive design on purpose; list the failure mode and the fix  
5. Add two rows to your personal glossary with examples from work  
6. Produce a one-page cheat sheet you could use in an interview  


### Mini case study

**Scenario:** Leadership wants this capability in production in six weeks with two engineers.

**Your job:** Propose an MVP that keeps irreversible risks controlled, names the eval gates, and lists what you explicitly defer.

Deliverable structure:
- MVP user story  
- Non-goals  
- Architecture (6 boxes max)  
- Eval gate table  
- Risk register (top 5)  
- Week-by-week plan  


In [ ]:
# Case study helper — risk register
import pandas as pd

risks = pd.DataFrame([
    {'risk': 'quality_miss', 'likelihood': 3, 'impact': 3, 'mitigation': 'golden evals + canary'},
    {'risk': 'cost_overrun', 'likelihood': 3, 'impact': 2, 'mitigation': 'budgets + cache'},
    {'risk': 'data_leak', 'likelihood': 2, 'impact': 5, 'mitigation': 'ACL + redaction'},
    {'risk': 'prompt_injection', 'likelihood': 4, 'impact': 4, 'mitigation': 'boundaries + allowlists'},
    {'risk': 'ops_pages', 'likelihood': 3, 'impact': 3, 'mitigation': 'runbooks + rollback'},
])
risks['score'] = risks.likelihood * risks.impact
print(risks.sort_values('score', ascending=False).to_string(index=False))


### Interview drill (topic-local)

Use the STAR or design template. Timebox 8 minutes.

**Prompt:** “Walk me through how you would productionize the main idea of this notebook.”

Checklist for a strong answer:
- [ ] Clarifying questions  
- [ ] Constraints & numbers  
- [ ] Diagram  
- [ ] Deep dive on hardest part  
- [ ] Evals  
- [ ] Security  
- [ ] Rollout / rollback  


### Glossary boost

| Term | Expanded meaning |
|------|------------------|
| Canary | Partial traffic to a new variant with automatic rollback |
| Golden set | Versioned labeled examples for regression |
| TTFT | Time to first token — interactive UX driver |
| Packing | Selecting/ordering context under a token budget |
| HITL | Human approval inserted before side effects |
| Idempotency | Safe retries without duplicate side effects |
| Shadow traffic | New system sees traffic but doesn't affect users |
| Circuit breaker | Stop calling a failing dependency temporarily |


In [ ]:
# Self-check quiz (run and answer mentally before printing answers)
QUESTIONS = [
    'What oracle proves success for this topic?',
    'Name one metric that can be gamed and a better alternative.',
    'What is the top security failure mode?',
    'What would you defer in an MVP?',
    'How do you rollback a bad change here?',
]
for i, q in enumerate(QUESTIONS, 1):
    print(f'Q{i}. {q}')
print('\n--- suggested answer hints ---')
HINTS = [
    'executable tests / task success / human rubric',
    'longer answers != better; use task success',
    'trust boundary crossing / injection / ACL',
    'multi-agent, perfect UI, every connector',
    'versioned prompts/models + traffic switch',
]
for h in HINTS:
    print('-', h)


### Further practice roadmap for `07-coding-rag`

| Horizon | Action |
|---------|--------|
| Today | Re-run all code cells; note questions |
| This week | Apply one technique to a real repo/service |
| This month | Add an eval or security test covering this topic |
| Interview ready | Give a 10-minute teach-back with a diagram |


## Lab: End-to-end scenario

Work this scenario in your notes, then implement the smallest possible spike.

### Scenario brief
A team wants to adopt the techniques from this notebook for a **real internal tool** used daily by 200 people. Leadership cares about reliability and auditability more than flashy demos.

### Deliverables
1. One-paragraph problem statement  
2. Success metrics (3) with oracles  
3. Architecture sketch with trust boundaries  
4. Threats / failure modes (5)  
5. Eval plan (offline + online)  
6. 2-week MVP scope and explicit non-goals  

### Review questions
- What happens when context is empty?  
- What happens when the model is down?  
- What happens when a user is malicious?  
- How do you prove a release is safer/better than last week?  


In [ ]:
# Lab helper — MVP scope tracker
from dataclasses import dataclass, field

@dataclass
class MVP:
    must: list[str] = field(default_factory=list)
    should: list[str] = field(default_factory=list)
    defer: list[str] = field(default_factory=list)

    def show(self):
        for label, items in [('MUST', self.must), ('SHOULD', self.should), ('DEFER', self.defer)]:
            print(label)
            for i in items:
                print(' -', i)

mvp = MVP(
    must=['core happy path', 'authn', 'basic eval smoke', 'rollback switch'],
    should=['streaming UX', 'dashboards'],
    defer=['multi-agent', 'perfect personalization', 'every connector'],
)
mvp.show()


## Operator runbook sketch

| Symptom | Likely cause | First checks | Mitigation |
|---------|--------------|--------------|------------|
| Latency spike | Downstream model / retrieve | p95 by stage, saturation | shed load, failover |
| Quality drop | Prompt/model/index change | diff versions, eval slice | rollback |
| Cost spike | loops / huge prompts | tokens/req, step counts | budget breaker |
| Security alert | injection / ACL | traces + retrieved IDs | kill switch |

Keep this table in your ops wiki; customize per system.


In [ ]:
# Operator helper — stage latency rollup
from statistics import mean

stages = {
    'gateway': [20, 25, 22],
    'retrieve': [80, 120, 95],
    'generate': [900, 1100, 980],
}
for k, v in stages.items():
    print(f'{k:10} mean={mean(v):.0f}ms max={max(v)}ms')
print('e2e~', sum(mean(v) for v in stages.values()), 'ms')


## Teaching notes (for study groups)

- Start with the comparison table; argue both sides for 5 minutes  
- Pair-program one demo cell with a real endpoint (placeholder keys)  
- Each person writes one failure case the suite must catch  
- End with a 60-second summary of when *not* to use the technique  


## Summary & Key Takeaways

- Code RAG needs structure-aware chunking and exact match
- Hybrid retrieval beats embeddings alone for identifiers
- Pack with citations and budgets; expand via graph when needed
